In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import json
import pickle
import numpy as np
import pandas as pd


In [2]:
#for good data sets, only thing to change
data_set_name = 'weinreb_all_genes'

#input
hyper_params_input = data_set_name + '_test_accuracy_best_hyper_params.json'
model_weights_input = data_set_name + '_model_weights.pth'
test_data_set_input = data_set_name + '_test_dataset.pth'
data_set_input = data_set_name + '.pkl'
barcode_NN_cont_mapping_input = data_set_name + '_barcode_NN_cont_mapping.pkl'

#output
confusion_matrix_output = data_set_name + 'confusion_matrix.csv'
classification_report_output = data_set_name + 'classification_report.csv'



In [8]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, features=None, labels=None):
        if features is not None and labels is not None:
            self.labels =labels
            self.features=features
            self.dataframe = dataframe
        else:
            self.dataframe = dataframe
            self.labels = self.dataframe['barcode_NN'].values
            self.features = self.dataframe.drop(['barcode_cont', 'barcode_NN'], axis=1).values

        
    def __len__(self):
        """
        Returns the total number of samples in the dataset.
        """
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        sample = torch.tensor(self.features[idx], dtype=torch.float)  
        label = torch.tensor(self.labels[idx], dtype=torch.long)  
        return sample, label
    
    def num_classes(self):
        """
        Returns the number of unique classes in the dataset.
        """
        return np.unique(self.labels).size
        
    def num_features(self):
        """
        Returns the number of features for each sample in the dataset.
        """
        return self.features.shape[1]  
    


In [10]:
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class CustomLayer(nn.Module):
    
    def __init__(self, size, dropout_rate, activation_func_name):
        super(CustomLayer, self).__init__()
        self.linear = nn.Linear(size, size)
        self.bn = nn.BatchNorm1d(size)
        self.dropout = nn.Dropout(dropout_rate)
        self.activation_func_name = activation_func_name  

        # Selecting the activation function
        if activation_func_name == 'relu':
            self.activation_func = nn.ReLU()
        elif activation_func_name == 'elu':
            self.activation_func = nn.ELU()
        elif activation_func_name == 'selu':
            self.activation_func = nn.SELU()
        elif activation_func_name == 'swish':
            self.activation_func = Swish()
        elif activation_func_name == 'mish':
            self.activation_func = nn.Mish()
        else:
            raise ValueError("Unsupported activation function")

        self.initialize_weights() 
            
    def initialize_weights(self):
        if self.activation_func_name in ['relu', 'swish', 'mish']:
            nn.init.kaiming_normal_(self.linear.weight, mode='fan_out', nonlinearity='relu')
        elif self.activation_func_name in ['elu', 'selu']:
            nn.init.kaiming_normal_(self.linear.weight, mode='fan_out', nonlinearity='leaky_relu')
        nn.init.constant_(self.linear.bias, 0)
         
    def forward(self, x):
        identity = x
        out = self.linear(x)

        out = self.bn(out)
        out = self.activation_func(out)
        out = self.dropout(out)
        out += identity  
        return out

class GeneExpressionModel(nn.Module):
         
    def __init__(self, input_dimension, num_classes, hidden_units, dropout_rate, num_layers, activation_function):
        super(GeneExpressionModel, self).__init__()

        # First layer to transform input dimension to hidden_units dimension
        self.initial_layer = nn.Linear(input_dimension, hidden_units)

        # Create a list of custom layers with residual connections
        self.layers = nn.ModuleList([CustomLayer(hidden_units, dropout_rate, activation_function) for _ in range(num_layers - 1)])
        
        # Output layer
        self.output = nn.Linear(hidden_units, num_classes) 

        with torch.no_grad():
            # Xavier initialization for the initial layer
            nn.init.kaiming_normal_(self.initial_layer.weight)
            nn.init.constant_(self.initial_layer.bias, 0)
           
            # Scaled Kaiming initialization for the output layer
            nn.init.kaiming_normal_(self.output.weight, mode='fan_out', nonlinearity='relu')
            self.output.weight.data *= 0.01  # Scaling factor
            nn.init.constant_(self.output.bias, 0)  

    def forward(self, x):
        # Pass the input through the initial layer
        x = self.initial_layer(x)

        # Pass the input through each custom layer
        for layer in self.layers:
            x = layer(x)

        # Output layer
        x = self.output(x)
        return x


Load the best neural network and the test data set that we will analyze.

In [16]:
with open(hyper_params_input, 'r') as f:
    best_hyperparameters = json.load(f)

# initialize model and data loaders
best_model = GeneExpressionModel(
    input_dimension=best_hyperparameters["data_overview"]["number_of_genes"],
    num_classes=best_hyperparameters["data_overview"]["number_of_barcodes"],
    hidden_units=best_hyperparameters["hyperparameters"]["hidden_units"],
    dropout_rate=best_hyperparameters["hyperparameters"]["dropout_rate"],
    num_layers=best_hyperparameters["hyperparameters"]["num_layers"],
    activation_function=best_hyperparameters["hyperparameters"]["activation_function"]
)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

test_dataset = torch.load(test_data_set_input)
best_model.load_state_dict(torch.load(model_weights_input))
best_model.to(device)



Here code for confusion matrix starts

In [ ]:
def evaluate_model_and_collect_labels(model, data_loader):
    model.eval()
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for data, targets in data_loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())
    return all_predictions, all_labels



In [ ]:

test_loader = DataLoader(test_dataset, batch_size=200, shuffle=False)  
predictions, labels = evaluate_model_and_collect_labels(best_model, test_loader)
# Load the mapping dictionary
with open(barcode_NN_cont_mapping_input, 'rb') as f:
    barcode_NN_to_cont_mapping = pickle.load(f)

# Function to remap labels or predictions
def remap_labels_or_predictions(items, mapping):
    return [mapping[item] for item in items]

# Remap labels and predictions
converted_labels = remap_labels_or_predictions(labels, barcode_NN_to_cont_mapping)
converted_predictions = remap_labels_or_predictions(predictions, barcode_NN_to_cont_mapping)


conf_matrix = confusion_matrix(converted_labels, converted_predictions)
class_names = [f'Barcode_Cont {barcode}' for barcode in sorted(set(converted_labels + converted_predictions))]
df_conf_matrix = pd.DataFrame(conf_matrix, index=class_names, columns=class_names)
df_conf_matrix.to_csv(confusion_matrix_output, index=True)
report = classification_report(converted_labels, converted_predictions, target_names=class_names, output_dict=True)
df_report = pd.DataFrame(report).transpose()
df_report.to_csv(classification_report_output, index=True)


